In [1]:
pip install sentence-transformers faiss-cpu

   ---------------------------------------- 0.0/571.3 kB ? eta -:--:--
   ---------------------------------------- 571.3/571.3 kB 12.3 MB/s  0:00:00
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ------------ --------------------------- 3.1/10.4 MB 17.8 MB/s eta 0:00:01
   ---------------- ----------------------- 4.2/10.4 MB 10.8 MB/s eta 0:00:01
   ------------------ --------------------- 4.7/10.4 MB 7.4 MB/s eta 0:00:01
   ------------------- -------------------- 5.0/10.4 MB 5.9 MB/s eta 0:00:01
   -------------------- ------------------- 5.2/10.4 MB 5.3 MB/s eta 0:00:01
   ---------------------- ----------------- 5.8/10.4 MB 4.5 MB/s eta 0:00:02
   ----------------------- ---------------- 6.0/10.4 MB 4.2 MB/s eta 0:00:02
   ------------------------ --------------- 6.3/10.4 MB 3.9 MB/s eta 0:00:02
   ------------------------- -------------- 6.6/10.4 MB 3.4 MB/s eta 0:00:02
   ------------------------- -------------- 6.6/10.4 MB 3.4 MB/s eta 0:00:02
   ------

In [2]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd


df_final_clean = pd.read_csv("data_clean.csv") 
print("✅ Data Loaded Successfully!")

# 1. Step: Text Enrichment (Critical for RAG)
# Hum description aur specs ko combine kar rahe hain taake search accurate ho
def create_search_string(row):
    text = (f"Industrial Part: {row['DESCRIPTION']}. "
            f"Rated Current: {row['Rated Current (A)']}A. "
            f"Voltage: {row['Rated Voltage (V)']}V. "
            f"Material: {row['Material']}. "
            f"Physical Size: {row['Product Diameter']}mm x {row['Product Length']}mm.")
    return text.replace('NOT SPECIFIED', '').strip()

df_final_clean['search_context'] = df_final_clean.apply(create_search_string, axis=1)

# 2. Step: Load Embedding Model
# 'all-MiniLM-L6-v2' chota aur tez hai, local CPU/GPU ke liye best hai
print("⏳ Loading Model...")
model = SentenceTransformer('all-MiniLM-L6-v2') 

# 3. Step: Generate Vectors
print("⏳ Generating Embeddings (be patient)...")
embeddings = model.encode(df_final_clean['search_context'].tolist(), show_progress_bar=True)

# 4. Step: Initialize FAISS Index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension) # L2 distance for similarity
index.add(np.array(embeddings).astype('float32'))

# 5. Step: Save Everything
# Taake Phase 3 mein humein ye sab dubara na karna pade
df_final_clean.to_csv("processed_parts_for_rag.csv", index=False)
faiss.write_index(index, "parts_vector_db.index")

print(f"✅ Success! Vector Database created with {index.ntotal} parts.")

✅ Data Loaded Successfully!
⏳ Loading Model...


c:\Users\LAIBA\Desktop\Projects\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LAIBA\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3943.56it/s]


⏳ Generating Embeddings (be patient)...


Batches: 100%|██████████| 32/32 [00:14<00:00,  2.18it/s]

✅ Success! Vector Database created with 998 parts.


In [3]:
pip install ollama

  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 20.0 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)

   -------- ------------------------------- 1/5 [pydantic-core]
   ------------------------ --------------- 3/5 [pydantic]
   ------------------------ --------------- 3/5 [pydantic]
   ------------------------ --------------- 3/5 [pydantic]
   ------------------------ --------------- 3/5 [pydantic]
   ------------------------ --------------- 3/5 [pydantic]
   ------------------------ --------------- 3/5 [pydantic]
   ------------------------ --------------- 3/5 [pydantic]
   ------------------------ --------------- 3/5 [pydantic]
   ------------------------ --------------- 3/5 [pyda